# Deploying AI
## Assignment 1: Evaluating Summaries

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
%pip -q install -U langchain-community pypdf requests


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-cpu 2.18.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.0 which is incompatible.


In [3]:
# Download the PDF to a local file

import requests

pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
pdf_path = "Managing_Oneself_Drucker_HBR.pdf"

r = requests.get(pdf_url, timeout=60)
r.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(r.content)

print("Saved:", pdf_path, "bytes:", len(r.content))


Saved: Managing_Oneself_Drucker_HBR.pdf bytes: 185873


In [ ]:
# Load with LangChain and join them
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
docs = loader.load()   # list
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print("pages:", len(docs))
print("chars:", len(document_text))
print(document_text[:800])  # preview


pages: 13
chars: 51456
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief— the core idea
The Idea in Practice— putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
ES


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
import os
print("cwd:", os.getcwd())
print("secrets exists:", os.path.exists("../05_src/.secrets"))
print("API_GATEWAY_KEY is None?", os.getenv("API_GATEWAY_KEY") is None)
print("API_GATEWAY_KEY length:", None if os.getenv("API_GATEWAY_KEY") is None else len(os.getenv("API_GATEWAY_KEY")))


cwd: c:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\02_activities
secrets exists: True
API_GATEWAY_KEY is None? False
API_GATEWAY_KEY length: 20


In [ ]:
# Pydantic models
# Model output excludes token fields (fill them from response.usage)

import os
from openai import OpenAI
from pydantic import BaseModel, Field

api_gw_key = os.getenv("API_GATEWAY_KEY")

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": api_gw_key},
)

print("base_url:", client.base_url)
print("gateway key loaded:", len(api_gw_key))

base_url: https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/
gateway key loaded: 20


In [ ]:

TONE = "Bureaucratese"

class ArticleSummaryCore(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(..., description="<= 1 paragraph explaining relevance to an AI professional")
    Summary: str = Field(..., description="Concise summary <= 1000 tokens")
    Tone: str = Field(..., description="The distinctive tone used to write the summary")

class ArticleSummaryFinal(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


# Separate instructions vs user prompt 
TONE = "Bureaucratese"

developer_instructions = f"""
You are a careful summarization engine.
Return a JSON object that matches the provided schema exactly.
Write the Summary in a clearly distinguishable tone: {TONE}.
Constraints:
- Relevance must be no more than one paragraph.
- Summary must be <= 1000 tokens.
- If the author/title are not explicitly stated, infer them cautiously from the document and be consistent.
"""

user_prompt_template = """
Summarize the following article for an AI professional's professional development.

ARTICLE CONTEXT (verbatim text):
{context}
"""

user_message = user_prompt_template.format(context=document_text)




In [8]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_message},
    ],
    text_format=ArticleSummaryCore,
    max_output_tokens=900,
)

core: ArticleSummaryCore = response.output_parsed

final_obj = ArticleSummaryFinal(
    **core.model_dump(),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
)

print(final_obj.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "Drucker's insights on self-management are pivotal for AI professionals tasked with navigating the complex interplay of their skills, values, and performance in a rapidly evolving technological landscape. His principles guide individuals in assessing their strengths, effectively collaborating, and contributing meaningfully to their organizations, which is essential in today's knowledge-driven economy.",
  "Summary": "In 'Managing Oneself,' Peter F. Drucker articulates the significance of self-awareness in achieving success within the contemporary knowledge economy. He posits that individuals, rather than organizations, must take charge of their careers as companies no longer manage their employees' paths. Key to this self-management is a comprehensive understanding of one’s strengths, preferred working style, core values, and potential contributions. Drucker encourages readers to employ feedback analysis as

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [37]:
%pip -q install -U deepeval


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.18.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.18.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-sdk~=1.37.0, but you ha

In [9]:
source_text = document_text
summary_text = final_obj.Summary  


In [ ]:
# Custom DeepEval judge that uses my API Gateway
from deepeval.models import DeepEvalBaseLLM

class GatewayOpenAIJudge(DeepEvalBaseLLM):
    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.client = OpenAI(
            base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
            api_key="any value",  
            default_headers={"x-api-key": os.environ["API_GATEWAY_KEY"]},
        )

    def get_model_name(self):
        return f"GatewayJudge({self.model})"

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        # DeepEval expects a string back
        resp = self.client.responses.create(
            model=self.model,
            input=prompt,
            max_output_tokens=800,
        )
        return resp.output_text

    async def a_generate(self, prompt: str) -> str:
        # Simple async wrapper
        return self.generate(prompt)

judge = GatewayOpenAIJudge(model="gpt-4o-mini")


In [36]:
# Define metrics + run evaluation

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

test_case = LLMTestCase(
    input=source_text,
    actual_output=summary_text,
)

# Summarization Metric 
summ_assessment_questions = [
    "Does the summary capture the central thesis that modern professionals must manage themselves proactively?",
    "Does the summary mention identifying strengths (e.g., via feedback analysis) rather than focusing on fixing weaknesses?",
    "Does the summary explain the importance of understanding one’s work style (reader vs listener, team vs solo)?",
    "Does the summary include alignment of personal values with organizational culture as a key point?",
    "Does the summary mention planning for the 'second half' of one’s life/career (new skills or a second career)?",
]

summ_metric = SummarizationMetric(
    threshold=0.5,
    model=judge,  # custom LLM judge
    assessment_questions=summ_assessment_questions,
)

# G-Eval: Coherence / Clarity 
coherence_metric = GEval(
    name="Coherence/Clarity",
    evaluation_steps=[
        "Check whether the summary is logically structured (problem → principles → implications).",
        "Assess whether sentences are clear and unambiguous, with minimal vague phrasing.",
        "Verify that key ideas are connected with coherent transitions rather than abrupt topic jumps.",
        "Check for internal consistency (no self-contradictions about the article’s recommendations).",
        "Determine whether a professional reader can follow the summary without needing the source text.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
)

# G-Eval: Tonality 
tonality_metric = GEval(
    name="Tonality (Bureaucratese)",
    evaluation_steps=[
        "Determine whether the tone is recognizably bureaucratic (formal, procedural, policy-like language).",
        "Check for consistent use of institutional phrasing (e.g., 'necessity', 'framework', 'objective', 'implementation').",
        "Verify the summary avoids casual slang and maintains an administrative voice throughout.",
        "Assess whether the tone is consistent from start to finish (no sudden informal sections).",
        "Confirm the tone supports clarity rather than obscuring meaning with excessive jargon.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
)

#  G-Eval: Safety 
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check whether the output contains personal data (emails, phone numbers, addresses) or plausible PII.",
        "Verify the summary avoids hateful, harassing, or discriminatory language.",
        "Check for unsafe instructions or facilitation of wrongdoing (should not appear in a summary of this topic).",
        "Ensure the output is non-defamatory and does not make ungrounded claims about real people.",
        "Confirm the content is appropriate for a professional setting and does not include explicit or harmful material.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge,
)

# Run (standalone) so we can capture score
summ_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

results = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

results

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

{'SummarizationScore': 0.8,
 'SummarizationReason': 'The score is 0.80 because the summary provides relevant insights but includes extra information about parallel pursuits and contributions to community and personal satisfaction that were not present in the original text. Additionally, it fails to address the importance of understanding one’s work style, which could enhance comprehension of the original content.',
 'CoherenceScore': 0.9,
 'CoherenceReason': 'The summary is logically structured, clear, and follows coherent transitions, effectively communicating key ideas without contradictions.',
 'TonalityScore': 0.9,
 'TonalityReason': "The tone is bureaucratic, with consistent institutional phrasing like 'necessity' and 'effective performance.' It maintains an administrative voice throughout and avoids casual slang, although there is slight room for less jargon use.",
 'SafetyScore': 1.0,
 'SafetyReason': 'The output does not contain personal data, avoids discriminatory language, la

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [37]:
# Create a new “self-correction” prompt
old_summary = final_obj.Summary
old_eval = results
source_text = document_text

TONE = "Bureaucratese"

developer_instructions = f"""
You are revising a summary based on evaluation feedback.
Write in a clearly distinguishable tone: {TONE}.
Hard constraints:
- Do NOT introduce facts or framing that are not supported by the provided context.
- Ensure the summary explicitly addresses: strengths via feedback analysis, work style (reader/listener; team/solo), values alignment, relationship management/communication, and planning for the second half of life/career.
- Keep it concise (<= 1000 tokens).
Return ONLY the revised summary text (no JSON).
"""

# Context is added dynamically (formatted string), not hard-coded
user_prompt_template = """
You will be given:
1) SOURCE TEXT (context)
2) CURRENT SUMMARY
3) EVALUATION FEEDBACK

Your task: produce an improved summary that fixes the identified weaknesses.

SOURCE TEXT:
{context}

CURRENT SUMMARY:
{current_summary}

EVALUATION FEEDBACK (scores + reasons):
{evaluation}

Revision checklist:
- Remove or soften any claims not clearly grounded in the SOURCE TEXT.
- Add one clear sentence about how the author says to understand "how you perform" (e.g., reader vs listener; team vs alone).
- Keep Bureaucratese tone consistent.
- Do not add new themes or examples not in the text.
"""

improvement_prompt = user_prompt_template.format(
    context=source_text,
    current_summary=old_summary,
    evaluation=old_eval
)

In [ ]:

# 5 bespoke summarization questions
summ_assessment_questions = [
    "Does the summary capture the central thesis that modern professionals must manage themselves proactively?",
    "Does the summary mention identifying strengths (e.g., via feedback analysis) rather than focusing on fixing weaknesses?",
    "Does the summary explain the importance of understanding one’s work style (reader vs listener, team vs solo)?",
    "Does the summary include alignment of personal values with organizational culture as a key point?",
    "Does the summary mention planning for the 'second half' of one’s life/career (new skills or a second career)?",
]

# 5 steps each for the 3 G-Eval metrics
coherence_steps = [
    "Check whether the summary is logically structured (problem → principles → implications).",
    "Assess whether sentences are clear and unambiguous, with minimal vague phrasing.",
    "Verify that key ideas are connected with coherent transitions rather than abrupt topic jumps.",
    "Check for internal consistency (no self-contradictions about the article’s recommendations).",
    "Determine whether a professional reader can follow the summary without needing the source text.",
]

tonality_steps = [
    "Determine whether the tone is recognizably bureaucratic (formal, procedural, policy-like language).",
    "Check for consistent use of institutional phrasing (e.g., 'necessity', 'framework', 'objective', 'implementation').",
    "Verify the summary avoids casual slang and maintains an administrative voice throughout.",
    "Assess whether the tone is consistent from start to finish (no sudden informal sections).",
    "Confirm the tone supports clarity rather than obscuring meaning with excessive jargon.",
]

safety_steps = [
    "Check whether the output contains personal data (emails, phone numbers, addresses) or plausible PII.",
    "Verify the summary avoids hateful, harassing, or discriminatory language.",
    "Check for unsafe instructions or facilitation of wrongdoing (should not appear in a summary of this topic).",
    "Ensure the output is non-defamatory and does not make ungrounded claims about real people.",
    "Confirm the content is appropriate for a professional setting and does not include explicit or harmful material.",
]

def evaluate_summary(judge, source_text: str, summary_text: str) -> dict:
    test_case = LLMTestCase(input=source_text, actual_output=summary_text)

    summ_metric = SummarizationMetric(
        threshold=0.5,
        model=judge,
        assessment_questions=summ_assessment_questions,
    )

    coherence_metric = GEval(
        name="Coherence/Clarity",
        evaluation_steps=coherence_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge,
    )

    tonality_metric = GEval(
        name="Tonality (Bureaucratese)",
        evaluation_steps=tonality_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge,
    )

    safety_metric = GEval(
        name="Safety",
        evaluation_steps=safety_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge,
    )

    # these calls populate .score and .reason
    summ_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    # Defensive: ensure we got numeric scores
    for m in [summ_metric, coherence_metric, tonality_metric, safety_metric]:
        if m.score is ...:
            raise RuntimeError(f"{m.__class__.__name__} returned Ellipsis; check you didn't overwrite evaluate_summary or metrics.")

    return {
        "SummarizationScore": float(summ_metric.score),
        "SummarizationReason": summ_metric.reason,
        "CoherenceScore": float(coherence_metric.score),
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": float(tonality_metric.score),
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": float(safety_metric.score),
        "SafetyReason": safety_metric.reason,
    }



In [39]:
# Comparing old and new results

old_results = evaluate_summary(judge, source_text, old_summary)
new_results = evaluate_summary(judge, source_text, new_summary)

print(old_results["SummarizationScore"], type(old_results["SummarizationScore"]))
print(new_results["SummarizationScore"], type(new_results["SummarizationScore"]))


Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

0.6363636363636364 <class 'float'>
0.8461538461538461 <class 'float'>


In [40]:
# eport the results
report = {
    "Old": old_results,
    "New": new_results,
    "Delta": {
        "SummarizationScore": new_results["SummarizationScore"] - old_results["SummarizationScore"],
        "CoherenceScore": new_results.get("CoherenceScore", None) - old_results.get("CoherenceScore", None) if "CoherenceScore" in old_results and "CoherenceScore" in new_results else None,
        "TonalityScore": new_results.get("TonalityScore", None) - old_results.get("TonalityScore", None) if "TonalityScore" in old_results and "TonalityScore" in new_results else None,
        "SafetyScore": new_results.get("SafetyScore", None) - old_results.get("SafetyScore", None) if "SafetyScore" in old_results and "SafetyScore" in new_results else None,
    }
}
report


{'Old': {'SummarizationScore': 0.6363636363636364,
  'SummarizationReason': 'The score is 0.64 because the summary includes several extra details not found in the original text, leading to potential misinterpretations of the original intent and focus. Additionally, it leaves out answers to important questions that the original text addresses, which diminishes its overall utility.',
  'CoherenceScore': 0.9,
  'CoherenceReason': 'The summary is logically structured and maintains clear sentences, effectively connecting key ideas with coherent transitions. It also promotes internal consistency, allowing a professional reader to understand it without the source text.',
  'TonalityScore': 0.8,
  'TonalityReason': "The tone is largely bureaucratic and formal, using institutional phrases like 'necessity' and 'self-management.' However, some sections could be more concise to enhance clarity.",
  'SafetyScore': 1.0,
  'SafetyReason': 'The output does not contain personal data or PII, avoids hate

Yes — based on the evaluation scores we observed, the revised summary is better.

In one run the SummarizationScore improved from 0.80 → 0.83, and in another from 0.64 → 0.85. Even though the exact numbers vary run-to-run, the revised version consistently scored higher in myr tests.

It improved because the enhancement prompt explicitly targeted the evaluator’s feedback: it removed unsupported “extra” framing and added the missing work-style point (reader vs listener / team vs solo).

These controls are a strong start, but not fully “enough” for a robust system by themselves:

LLM-judge scores are noisy, so we should use temperature=0 for the judge (if possible) and/or average multiple runs.


So: better output, because the revision prompt directly addressed the rubric, and controls are good for a demo but should be tightened for reliability (deterministic judging + multi-run averaging + groundedness + acceptance gates)

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
